# Customer Churn Prediction using Telco Customer Churn Dataset

**Course:** MIS 308 Data Science  
**Project Type:** Machine Learning Classification  

The aim of this project is to analyze customer data and develop machine learning classification models to predict whether a customer is likely to leave the service (churn).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay

sns.set_theme(style="whitegrid")

## 1. Load Dataset

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

In [ ]:
df.shape, df.info()

## 2. Data Cleaning

The `TotalCharges` column is stored as object type because it contains blank values. It is converted to numeric format and missing values are removed. The `customerID` column is removed because it does not provide predictive information.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing values in TotalCharges:", df["TotalCharges"].isna().sum())
df = df.dropna().copy()
df = df.drop("customerID", axis=1)
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df.head()

## 3. Exploratory Data Analysis

In [ ]:
df["Churn"].value_counts(normalize=True).mul(100).round(2)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Churn", color="steelblue")
plt.title("Churn Distribution")
plt.xticks([0,1], ["No Churn", "Churn"])
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df, x="Contract", hue="Churn", palette="Set2")
plt.title("Churn by Contract Type")
plt.legend(title="Churn", labels=["No", "Yes"])
plt.show()

In [ ]:
plt.figure(figsize=(8,4.5))
sns.countplot(data=df, x="PaymentMethod", hue="Churn", palette="Set2")
plt.title("Churn by Payment Method")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Churn", labels=["No", "Yes"])
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure Distribution by Churn")
plt.xticks([0,1], ["No Churn", "Churn"])
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges by Churn")
plt.xticks([0,1], ["No Churn", "Churn"])
plt.show()

## 4. Feature Engineering

In [ ]:
df["AvgChargesPerTenure"] = df["TotalCharges"] / (df["tenure"] + 1)
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 months", "13-24 months", "25-48 months", "49-72 months"]
)
df[["tenure", "TotalCharges", "AvgChargesPerTenure", "TenureGroup"]].head()

## 5. Train-Test Split and Preprocessing

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 6. Model Development

Four classification algorithms are implemented: Logistic Regression, Decision Tree, Random Forest, and K-Nearest Neighbors.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=6, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=200, max_depth=8, class_weight="balanced"),
    "KNN": KNeighborsClassifier(n_neighbors=15)
}

fitted_models = {}
for name, clf in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", clf)
    ])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    print(f"{name} trained successfully.")

## 7. Model Evaluation

In [ ]:
results = []

for name, model in fitted_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).sort_values(["F1 Score", "ROC-AUC"], ascending=False)
results_df

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("Best model:", best_model_name)
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title(f"ROC Curve - {best_model_name}")
plt.show()

## 8. Conclusion

This project developed and compared multiple classification models to predict customer churn. The analysis showed that churn is related to contract type, tenure, payment method, and monthly charges. Since churn prediction is a business-oriented problem, the best model was selected by considering F1-score and ROC-AUC rather than accuracy only.